# Universal PDF RAG Chatbot — LangChain + LangGraph + ChromaDB + Ollama (local)

This notebook is designed for **any PDF you provide**, not for one hard-coded book.

### To use another PDF
1. Put the PDF in the same folder as this notebook (or give its full path).
2. Change **only `PDF_PATH`** in the configuration cell.
3. Run all cells from top to bottom.

The notebook automatically:
- loads the selected PDF
- detects pages and basic chapter/section headings when present
- splits the document into chunks
- creates embeddings
- builds a fresh ChromaDB collection for that PDF
- uses LangGraph for conversational RAG
- remembers follow-up questions during the current notebook session
- retrieves relevant pages before generating an answer
- shows the source pages used for the answer

If a PDF has no chapter headings, the chatbot still works normally as a general PDF question-answering system.

In [1]:
# All packages installed in ONE pip call so the resolver picks
# versions that satisfy every constraint together.
!pip install -q -U --user \
    langchain langchain-community langchain-core langchain-text-splitters \
    langchain-ollama langchain-chroma langgraph pypdf \
    "huggingface_hub==0.23.4" \
    "transformers==4.41.2" \
    "tokenizers==0.19.1" \
    "sentence-transformers==2.7.0" \
    "langchain-huggingface==0.0.3"


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sanity check: confirm the pinned versions actually installed.
# Expected: transformers 4.41.2, tokenizers 0.19.1, sentence-transformers 2.7.0, huggingface_hub 0.23.4
import transformers, tokenizers, sentence_transformers, huggingface_hub
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

transformers: 4.41.2
tokenizers: 0.19.1
sentence-transformers: 2.7.0
huggingface_hub: 0.23.4


In [3]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silences Chroma telemetry error spam
import re
import json
import shutil
import hashlib
import pandas as pd
from pathlib import Path
from typing import TypedDict, Annotated

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print('Imports loaded successfully.')

Imports loaded successfully.


## 1. Configuration — change only `PDF_PATH` for another PDF

In [4]:
# ============================================================
# CHANGE ONLY THIS PATH WHEN YOU WANT TO USE ANOTHER PDF
# ============================================================
PDF_PATH = "ml_ebook.pdf"

# Local Ollama model. Make sure you've run `ollama pull llama3.2:3b` first.
OLLAMA_MODEL = "llama3.2:3b"  # good balance of speed/quality for 8GB RAM
OLLAMA_NUM_CTX = 4096  # explicit: prevents retrieved context from being silently truncated

# Retrieval settings
TOP_K = 8
FETCH_K = 24
MMR_LAMBDA = 0.5  # tested 0.5 vs 0.75: no measurable difference on this book

# Chunk settings
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# Embedding model
# MiniLM: fast and local. all-mpnet-base-v2 was tested and gave no clear gain here.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

path = Path(PDF_PATH)
if not path.exists():
    raise FileNotFoundError(
        f"PDF not found: {path.resolve()}\n"
        "Put the PDF in the notebook folder or set PDF_PATH to its full path."
    )
if path.suffix.lower() != ".pdf":
    raise ValueError("PDF_PATH must point to a .pdf file.")

print(f"Selected PDF: {path.name}")
print(f"Location: {path.resolve()}")

Selected PDF: ml_ebook.pdf
Location: D:\Codes\ML\ml_ebook.pdf


## 2. API key (not needed -- Ollama runs locally)

In [5]:
# No API key needed — Ollama runs entirely locally.
# Just make sure the Ollama app is running and you've pulled the model:
#   ollama pull llama3.2:3b
print("Using local Ollama model — no API key required.")


Using local Ollama model — no API key required.


## 3. Load the selected PDF

In [6]:
# Load PDF
loader = PyPDFLoader(str(path))
raw_pages = loader.load()

# A page is "table of contents" if most of its lines end in a page number
# after a run of spaces or dot leaders.
TOC_LINE = re.compile(r"(\s{4,}|(?:\.\s?){4,})\d{1,4}\s*$")

def is_toc_page(text, threshold=0.4):
    lines = [l for l in text.splitlines() if l.strip()]
    if len(lines) < 5:
        return False
    hits = sum(1 for l in lines if TOC_LINE.search(l))
    return hits / len(lines) >= threshold

# Keep the original PDF page number BEFORE removing empty pages.
pages = []
skipped_toc_pages = []
pdf_id = hashlib.sha256(path.read_bytes()).hexdigest()[:12]

for doc in raw_pages:
    text = doc.page_content

    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="ignore")
    elif not isinstance(text, str):
        text = str(text)

    # Remove null characters that can cause tokenizer/embedding problems.
    text = text.replace("\x00", " ")
    # Drop lone UTF-16 surrogates (they appear where a PDF has math symbols).
    # They cannot be encoded as UTF-8 and silently crash Chroma ingestion.
    text = text.encode("utf-8", errors="ignore").decode("utf-8").strip()

    if not text:
        continue

    # Skip table-of-contents pages: they repeat every topic name and crowd
    # real content out of the retrieved chunks.
    if is_toc_page(text):
        skipped_toc_pages.append(doc.metadata.get("page"))
        continue

    original_page = doc.metadata.get("page")
    if isinstance(original_page, int):
        page_number = original_page + 1
    else:
        page_number = len(pages) + 1

    doc.page_content = text
    doc.metadata["page"] = page_number
    doc.metadata["source_file"] = path.name
    doc.metadata["pdf_id"] = pdf_id
    pages.append(doc)

print(f"Loaded {len(pages)} non-empty text pages from {path.name}.")
if skipped_toc_pages:
    print(f"Skipped {len(skipped_toc_pages)} table-of-contents page(s) (0-based indices): {skipped_toc_pages}")


Loaded 266 non-empty text pages from ml_ebook.pdf.
Skipped 5 table-of-contents page(s) (0-based indices): [4, 5, 6, 7, 8]


## 4. Detect basic chapter/section headings automatically

This is intentionally **generic**. It does not assume a particular book title, author, or number of chapters.

If the PDF contains headings such as `Chapter 1`, `CHAPTER ONE`, or `Chapter 1: Introduction`, they are stored as metadata. If it does not, `chapter` simply remains `None`; normal RAG still works.

In [7]:
NUMBER_WORDS = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
    "nineteen": 19, "twenty": 20
}
ROMAN_VALUES = {"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}

def roman_to_int(value):
    total, previous = 0, 0
    for char in reversed(value.upper()):
        current = ROMAN_VALUES.get(char, 0)
        if current < previous:
            total -= current
        else:
            total += current
            previous = current
    return total

def detect_chapter(text):
    # Look near the beginning of a page so ordinary mentions of "chapter"
    # inside paragraphs are not treated as headings.
    lines = [re.sub(r"\s+", " ", x).strip() for x in text.splitlines()]
    lines = [x for x in lines if x]
    sample = lines[:12]
    for line in sample:
        m = re.match(r"^chapter\s*[:\-]?\s*(\d{1,3})\b", line, re.I)
        if m:
            return int(m.group(1)), line
        m = re.match(r"^chapter\s+([ivxlcdm]+)\b", line, re.I)
        if m:
            return roman_to_int(m.group(1)), line
        m = re.match(r"^chapter\s+(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty)\b", line, re.I)
        if m:
            return NUMBER_WORDS[m.group(1).lower()], line
    return None, None

chapter_starts = {}
for doc in pages:
    number, heading = detect_chapter(doc.page_content)
    doc.metadata["chapter"] = number
    doc.metadata["chapter_heading"] = heading
    if number is not None and number not in chapter_starts:
        chapter_starts[number] = doc.metadata["page"]

print(f"Detected chapters: {sorted(chapter_starts) if chapter_starts else 'No explicit chapter headings found'}")

Detected chapters: [1, 2, 3, 4, 5, 6, 7, 8, 9]


## 5. Split the PDF into retrieval chunks

In [8]:
# Split pages into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

clean_chunks = []

for i, doc in enumerate(chunks):
    text = doc.page_content

    if not isinstance(text, str):
        text = str(text)

    text = text.replace("\x00", " ").strip()

    if text:
        doc.page_content = text
        doc.metadata["chunk_id"] = i
        clean_chunks.append(doc)

chunks = clean_chunks

print(f"Created {len(chunks)} valid chunks.")

Created 635 valid chunks.


## 6. Create embeddings

In [9]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

print("Embedding model loaded successfully:", EMBEDDING_MODEL)

D:\Compiler\Python-3\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded successfully: sentence-transformers/all-MiniLM-L6-v2


## 7. Build a fresh ChromaDB for the selected PDF

The database directory is based on the **content hash of the PDF**, so changing the PDF automatically gives it a different database.

The current PDF's database is rebuilt when this notebook is run, which prevents old PDF chunks from accidentally mixing with the new PDF.

In [10]:
# Build ChromaDB (always a fresh collection for this exact PDF)

PDF_HASH = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
DB_DIR = Path("chroma_db") / PDF_HASH
COLLECTION_NAME = f"pdf_{PDF_HASH}"

# Release any Chroma client from a PREVIOUS run in this same kernel
# session. On Windows, sqlite keeps the DB file open/locked until the
# client object is actually closed and garbage-collected, which is
# exactly what caused the PermissionError on shutil.rmtree below.
import gc

if "vectorstore" in globals():
    try:
        vectorstore._client.reset()
    except Exception:
        pass
    del vectorstore

gc.collect()

# Guarantee a fresh DB: if this PDF was indexed before, wipe it first
# so stale chunks/metadata from an older run/schema can never linger.
# Retry with a short delay in case Windows hasn't released the file
# handle yet; if it's still locked after that, fall back to a new
# directory instead of crashing the whole run.
import time

if DB_DIR.exists():
    for attempt in range(5):
        try:
            shutil.rmtree(DB_DIR)
            break
        except PermissionError:
            time.sleep(1)
    else:
        DB_DIR = Path("chroma_db") / f"{PDF_HASH}_{int(time.time())}"
        print(f"Old DB directory was locked; using a fresh path: {DB_DIR}")

clean_docs = []

for doc in chunks:
    text = str(doc.page_content).encode("utf-8", errors="ignore").decode("utf-8").strip()

    if not text:
        continue

    # Make metadata Chroma-compatible (no None / nested types).
    metadata = {}
    for key, value in doc.metadata.items():
        if value is None:
            continue
        elif isinstance(value, (str, int, float, bool)):
            metadata[key] = value
        else:
            metadata[key] = str(value)

    clean_docs.append(Document(page_content=text, metadata=metadata))

print(f"Preparing {len(clean_docs)} chunks for embedding...")

vectorstore = Chroma.from_documents(
    documents=clean_docs,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_DIR)
)

print(f"Chroma collection: {COLLECTION_NAME}")
stored = vectorstore._collection.count()
print(f"Stored vectors: {stored}")

# Safety check: Chroma ingests in the background and can fail part-way without
# raising an error. If fewer vectors were stored than chunks prepared, part of
# the book is missing from the index -- stop instead of silently continuing.
if stored != len(clean_docs):
    raise RuntimeError(
        f"Only {stored} of {len(clean_docs)} chunks were stored in ChromaDB. "
        "Scroll up for an ingestion error, fix it, and re-run this cell."
    )

Preparing 635 chunks for embedding...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma collection: pdf_f50d04b9d0e6
Stored vectors: 635


## 8. Create the local Ollama model

In [11]:
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=OLLAMA_NUM_CTX
)

print(f"Ollama model: {OLLAMA_MODEL}")


Ollama model: llama3.2:3b


## 9. Create the retriever

MMR (Maximal Marginal Relevance) retrieves relevant chunks while reducing excessive duplication between chunks.

In [12]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K,
        "fetch_k": FETCH_K,
        "lambda_mult": MMR_LAMBDA
    }
)
print("Retriever ready.")

Retriever ready.


## 10. Helper functions for clean model output and source formatting

In [13]:
def clean_response(response):
    """Convert LangChain model content into clean printable text."""
    content = response.content if hasattr(response, "content") else response

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "text":
                    parts.append(str(item.get("text", "")))
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(x for x in parts if x).strip()

    return str(content).strip()


def format_context(documents):
    blocks = []
    for doc in documents:
        page = doc.metadata.get("page", "?")
        chapter = doc.metadata.get("chapter")
        heading = doc.metadata.get("chapter_heading")
        location = f"Page {page}"
        if chapter is not None:
            location += f" | Chapter {chapter}"
        if heading:
            location += f" | {heading}"
        blocks.append(f"[{location}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


def source_list(documents):
    seen = set()
    sources = []
    for doc in documents:
        page = doc.metadata.get("page", "?")
        key = (doc.metadata.get("source_file", path.name), page)
        if key not in seen:
            seen.add(key)
            sources.append(key)
    return sources

## 11. Conversational RAG prompt

The important part for follow-up questions is that the previous conversation is used to rewrite an incomplete question into a standalone question **before retrieval**.

In [14]:
rewrite_prompt = ChatPromptTemplate.from_template("""
You are rewriting a user's follow-up question for document retrieval.

Conversation history:
{history}

Latest user question:
{question}

Rewrite the latest question as one standalone search query.
- Preserve the user's intent.
- Resolve pronouns such as he, she, it, they, this, and that using the conversation.
- Do not answer the question.
- If the question is already standalone, return it unchanged.
- Return only the rewritten question.
""")

answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about the provided PDF.

Use ONLY the supplied PDF context to answer. Do not invent facts that are not supported by the context.

If the answer is not present in the retrieved context, your ENTIRE reply must be exactly:
"This information is not covered in the provided PDF."
In that case, do not add anything else -- no partial answer, no outside/general knowledge, no apology.

Give a COMPLETE and SPECIFIC answer, not just a related or partial one:
- Cover every part of the question, not just the part that is easiest to answer.
- Include concrete details from the context (definitions, examples, names, numbers) rather than a vague summary.
- Do not stop at a general statement when the context has a more precise answer available.

If the question asks you to compare, contrast, or state a difference between two or more things
(for example "difference between X and Y", "X vs Y", "how does X differ from Y"), you MUST
explicitly address EACH thing being compared, not just one side -- state what each one is before
comparing them, so neither side is left out.

Answer naturally and directly. For explanation questions, explain in simple language when appropriate.

PDF context:
{context}

Question:
{question}

Answer:
""")

## 12. LangGraph state and nodes

In [15]:
class RAGState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    standalone_question: str
    retrieved_docs: list
    answer: str
    sources: list


def rewrite_question(state: RAGState):
    messages = state.get("messages", [])
    latest = messages[-1].content if messages else ""
    previous = messages[:-1][-6:]

    history_text = "\n".join(
        f"{type(m).__name__}: {m.content}" for m in previous
    )

    if not previous:
        return {"standalone_question": latest}

    prompt = rewrite_prompt.format_messages(
        history=history_text,
        question=latest
    )
    rewritten = clean_response(llm.invoke(prompt))
    return {"standalone_question": rewritten or latest}


def safe_retrieve(question, k=None):
    # retriever.invoke() crashes INSIDE LangChain's own Document
    # construction if the Chroma collection has any corrupted record
    # (embedding + metadata stored but page_content missing) — a
    # try/except around retriever.invoke() can't catch that cleanly
    # because langchain builds Document objects internally before
    # returning control to us. So on failure, fall back to querying
    # the raw Chroma collection ourselves and skip any null-text hit
    # before constructing Documents at all.
    try:
        return retriever.invoke(question)
    except Exception:
        query_vector = embeddings.embed_query(question)
        raw = vectorstore._collection.query(
            query_embeddings=[query_vector],
            n_results=k or FETCH_K,
            include=["documents", "metadatas"]
        )
        docs = []
        for text, meta in zip(raw["documents"][0], raw["metadatas"][0]):
            if text and text.strip():
                docs.append(Document(page_content=text, metadata=meta or {}))
        return docs[: (k or TOP_K)]


def retrieve_documents(state: RAGState):
    question = state["standalone_question"]
    docs = safe_retrieve(question)

    return {
        "retrieved_docs": docs,
        "sources": source_list(docs)
    }


def generate_answer(state: RAGState):
    question = state["standalone_question"]
    docs = state.get("retrieved_docs", [])
    context = format_context(docs)

    prompt = answer_prompt.format_messages(
        context=context,
        question=question
    )
    answer = clean_response(llm.invoke(prompt))

    return {
        "answer": answer,
        "messages": [AIMessage(content=answer)]
    }

## 13. Build the LangGraph workflow

In [16]:
workflow = StateGraph(RAGState)
workflow.add_node("rewrite_question", rewrite_question)
workflow.add_node("retrieve_documents", retrieve_documents)
workflow.add_node("generate_answer", generate_answer)

workflow.add_edge(START, "rewrite_question")
workflow.add_edge("rewrite_question", "retrieve_documents")
workflow.add_edge("retrieve_documents", "generate_answer")
workflow.add_edge("generate_answer", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("LangGraph RAG workflow is ready.")

LangGraph RAG workflow is ready.


## 14. Ask questions programmatically

In [17]:
THREAD_ID = "pdf-chat-session"

def run_rag(question, thread_id=THREAD_ID):
    question = question.strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    return app.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"thread_id": thread_id}}
    )


def ask(question, thread_id=THREAD_ID, show_sources=True):
    result = run_rag(question, thread_id)

    answer = result.get("answer", "No answer generated.")

    print("\nAssistant:")
    print(answer)

    if show_sources:
        sources = result.get("sources", [])
        if sources:
            print("\nSources:")
            for filename, page in sources:
                print(f"- {filename} — page {page}")
        else:
            print("\nSources: No source pages returned.")

    return answer


## 15. Document verification

In [18]:
print("=" * 70)
print("PDF RAG READY")
print("=" * 70)
print(f"File:             {path.name}")
print(f"Pages loaded:     {len(pages)}")
print(f"Chunks created:   {len(chunks)}")
print(f"Chroma vectors:   {vectorstore._collection.count()}")
print(f"Detected chapters:{len(chapter_starts)}")
print(f"Database:         {DB_DIR}")
print("=" * 70)

PDF RAG READY
File:             ml_ebook.pdf
Pages loaded:     266
Chunks created:   635
Chroma vectors:   635
Detected chapters:9
Database:         chroma_db\f50d04b9d0e6


## 16. Interactive chatbot

Run this cell and ask questions about **whatever PDF you selected in `PDF_PATH`**.

Type `exit` or `quit` to stop.

In [19]:
print(f"\nChatbot ready for: {path.name}")
print("Type 'exit' or 'quit' to stop.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break
    ask(question)


Chatbot ready for: ml_ebook.pdf
Type 'exit' or 'quit' to stop.



You:  exit


Chat ended.


## 17. Example questions

These are examples only. The chatbot is not hard-coded to answer these questions.

Try questions such as:

- What is this document about?
- What is the main idea of this section?
- Explain a concept in simple language.
- What are the advantages and disadvantages of this approach?
- What is the difference between two concepts?
- What does the document say about a specific topic?
- Ask a follow-up question using "it", "they", "this", or "that".

For a quantitative test, use the **Evaluation** section below.


## 18. RAG Evaluation

This section evaluates the **current PDF and current retrieval settings**.

It measures:

1. **Retrieval Hit Rate** — whether the retriever returned at least one expected source page.
2. **Answer quality** — an LLM judge checks correctness and groundedness.

The included questions are based on the uploaded *Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow* PDF, especially Chapter 1. If you switch to another PDF, replace the evaluation questions with questions from that document.


In [20]:
# Evaluation dataset
# expected_pages are 1-BASED page numbers: the loader cell stores
# metadata["page"] = pdf_index + 1, i.e. the number you see in a PDF viewer's
# page box. The retrieval-hit-rate check compares against that value.
# (Verified against ml_ebook.pdf: all values below match the real content pages.)
#
# category is one of: "factual", "comparison", "multi_part", "out_of_document"

evaluation_data = [
    # ---------------- FACTUAL / DEFINITION (25) ----------------
    {
        "question": "What is Machine Learning?",
        "category": "factual",
        "expected_pages": [12],
        "reference_answer": "Machine Learning is the science (and art) of programming computers so they can learn from data. The book also defines it as a field of study that gives computers the ability to learn without being explicitly programmed."
    },
    {
        "question": "What is a training set?",
        "category": "factual",
        "expected_pages": [12],
        "reference_answer": "The examples a system uses to learn are called the training set. Each training example is called a training instance or sample."
    },
    {
        "question": "Why is Machine Learning useful for problems that require many hand-written rules?",
        "category": "factual",
        "expected_pages": [13, 14],
        "reference_answer": "A Machine Learning algorithm can learn useful patterns from examples, which makes the program shorter, easier to maintain, and often more accurate than a long list of hand-tuned rules, and it can automatically adapt when the patterns it is detecting change."
    },
    {
        "question": "What are the main criteria used to classify Machine Learning systems in Chapter 1?",
        "category": "factual",
        "expected_pages": [15],
        "reference_answer": "The book classifies systems by whether or not they are trained with human supervision, whether or not they can learn incrementally on the fly, and whether they work by comparing new data points to known data points or by detecting patterns and building a predictive model."
    },
    {
        "question": "What are the four major categories based on the amount and type of supervision?",
        "category": "factual",
        "expected_pages": [16],
        "reference_answer": "The four categories are supervised learning, unsupervised learning, semisupervised learning, and Reinforcement Learning."
    },
    {
        "question": "What is supervised learning?",
        "category": "factual",
        "expected_pages": [16],
        "reference_answer": "In supervised learning, the training data you feed to the algorithm includes the desired solutions, called labels."
    },
    {
        "question": "What is a typical classification task mentioned in the book?",
        "category": "factual",
        "expected_pages": [16],
        "reference_answer": "Spam filtering is a typical classification task: the system is trained with many example emails along with their class (spam or ham) and must learn how to classify new emails."
    },
    {
        "question": "What is regression in supervised learning?",
        "category": "factual",
        "expected_pages": [16, 17],
        "reference_answer": "Regression predicts a target numeric value, such as the price of a car, given a set of features (predictors) like mileage, age, and brand."
    },
    {
        "question": "What is unsupervised learning?",
        "category": "factual",
        "expected_pages": [18],
        "reference_answer": "In unsupervised learning, the training data is unlabeled, and the system tries to learn without a teacher."
    },
    {
        "question": "Name some unsupervised learning algorithms discussed in the book.",
        "category": "factual",
        "expected_pages": [18],
        "reference_answer": "Examples include K-Means, DBSCAN, Hierarchical Cluster Analysis (HCA), One-class SVM, Isolation Forest, PCA, Kernel PCA, LLE, t-SNE, Apriori, and Eclat."
    },
    {
        "question": "What is semisupervised learning?",
        "category": "factual",
        "expected_pages": [21],
        "reference_answer": "Semisupervised learning deals with partially labeled training data -- usually a lot of unlabeled data and a little bit of labeled data, as in photo-hosting services that cluster faces and only need one label per person."
    },
    {
        "question": "How does Reinforcement Learning work at a high level?",
        "category": "factual",
        "expected_pages": [22, 23],
        "reference_answer": "An agent observes the environment, selects and performs actions, and gets rewards or penalties in return; it must learn by itself the best strategy, called a policy, to get the most reward over time."
    },
    {
        "question": "What is batch learning?",
        "category": "factual",
        "expected_pages": [23],
        "reference_answer": "In batch learning, the system is incapable of learning incrementally: it must be trained using all the available data, generally offline, and then launched into production without learning anymore."
    },
    {
        "question": "What is online learning?",
        "category": "factual",
        "expected_pages": [24],
        "reference_answer": "In online learning, you train the system incrementally by feeding it data instances sequentially, either individually or in small groups called mini-batches, so it can learn about new data on the fly as it arrives."
    },
    {
        "question": "What is instance-based learning?",
        "category": "factual",
        "expected_pages": [26],
        "reference_answer": "Instance-based learning is when the system learns examples by heart and generalizes to new cases by using a similarity measure to compare them to the learned examples."
    },
    {
        "question": "What is model-based learning?",
        "category": "factual",
        "expected_pages": [27],
        "reference_answer": "Model-based learning is when you build a model from a set of examples and then use that model to make predictions on new cases."
    },
    {
        "question": "What are the two things that can go wrong in a Machine Learning project according to the book?",
        "category": "factual",
        "expected_pages": [32],
        "reference_answer": "The book summarizes them as \"bad algorithm\" and \"bad data,\" and starts by discussing examples of bad data."
    },
    {
        "question": "What is sampling bias?",
        "category": "factual",
        "expected_pages": [35],
        "reference_answer": "Sampling bias occurs when the sampling method used to build a training set is flawed and produces a nonrepresentative sample, as in the 1936 Literary Digest poll example the book gives. Nonresponse bias, where certain groups are less likely to respond, is described as a special type of sampling bias."
    },
    {
        "question": "What is overfitting in Machine Learning?",
        "category": "factual",
        "expected_pages": [36],
        "reference_answer": "Overfitting means the model performs well on the training data but does not generalize well, typically because it is too complex relative to the amount and noisiness of the training data."
    },
    {
        "question": "What is underfitting in Machine Learning?",
        "category": "factual",
        "expected_pages": [38],
        "reference_answer": "Underfitting is the opposite of overfitting: it occurs when the model is too simple to learn the underlying structure of the data, so it performs poorly even on the training examples."
    },
    {
        "question": "What is a hyperparameter?",
        "category": "factual",
        "expected_pages": [38],
        "reference_answer": "A hyperparameter is a parameter of the learning algorithm itself (not of the model). It is not affected by the learning algorithm and must be set prior to training, remaining constant during training."
    },
    {
        "question": "What is regularization?",
        "category": "factual",
        "expected_pages": [37],
        "reference_answer": "Regularization means constraining a model to make it simpler and reduce the risk of overfitting, and the amount of regularization to apply is controlled by a hyperparameter."
    },
    {
        "question": "What is the purpose of a test set?",
        "category": "factual",
        "expected_pages": [39],
        "reference_answer": "The test set is used to estimate the generalization error -- how well the model performs on new cases it has never seen -- by evaluating the trained model on data held out from training."
    },
    {
        "question": "What is the purpose of a validation set?",
        "category": "factual",
        "expected_pages": [40],
        "reference_answer": "A validation (holdout) set is held out from the training set so you can evaluate and compare several candidate models and choose the best one without touching the test set, keeping the test set's error estimate unbiased."
    },
    {
        "question": "What is the No Free Lunch theorem, as discussed in the book?",
        "category": "factual",
        "expected_pages": [41],
        "reference_answer": "The No Free Lunch theorem, from a paper by David Wolpert, states that if you make absolutely no assumptions about the data, there is no reason to prefer one model over any other -- the simplifications a model makes are themselves assumptions."
    },

    # ---------------- CONCEPTUAL / COMPARISON (15) ----------------
    {
        "question": "What is the difference between batch learning and online learning?",
        "category": "comparison",
        "expected_pages": [23, 24],
        "reference_answer": "Batch learning must be trained using all available data at once, generally offline, and does not learn further after launch; online learning trains incrementally, feeding the system data instances or mini-batches sequentially so it can adapt on the fly."
    },
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "category": "comparison",
        "expected_pages": [16, 18],
        "reference_answer": "Supervised learning uses training data that includes the desired labels, while unsupervised learning uses unlabeled data and the system tries to find structure on its own, without a teacher."
    },
    {
        "question": "What is the difference between instance-based and model-based learning?",
        "category": "comparison",
        "expected_pages": [26, 27],
        "reference_answer": "Instance-based learning generalizes by comparing new cases to memorized examples using a similarity measure, while model-based learning builds a general model from the examples and uses that model's parameters to predict on new cases."
    },
    {
        "question": "What is the difference between classification and regression?",
        "category": "comparison",
        "expected_pages": [16, 17],
        "reference_answer": "Classification predicts a discrete class, such as spam or ham, while regression predicts a continuous target numeric value, such as a car's price, from a set of predictors."
    },
    {
        "question": "What is the difference between a test set and a validation set?",
        "category": "comparison",
        "expected_pages": [39, 40],
        "reference_answer": "The validation (holdout) set is used repeatedly during development to compare candidate models and tune hyperparameters, while the test set is used only once, at the very end, to give an unbiased estimate of the final model's generalization error."
    },
    {
        "question": "What is the difference between holdout validation and cross-validation?",
        "category": "comparison",
        "expected_pages": [40, 41],
        "reference_answer": "Holdout validation sets aside a single validation set to evaluate candidate models; cross-validation instead uses multiple validation sets and averages the results across them, giving a more accurate performance measure at the cost of extra training time."
    },
    {
        "question": "What is the difference between feature selection and feature extraction?",
        "category": "comparison",
        "expected_pages": [36],
        "reference_answer": "Feature selection means selecting the most useful existing features to train on, while feature extraction means combining existing features to produce a new, more useful one (for example, via dimensionality reduction)."
    },
    {
        "question": "What is the difference between sampling bias and nonresponse bias?",
        "category": "comparison",
        "expected_pages": [35],
        "reference_answer": "Sampling bias is the general problem of a flawed sampling method producing a nonrepresentative training set; nonresponse bias is a special type of sampling bias where certain groups are underrepresented because they are less likely to respond, as in the 1936 Literary Digest poll."
    },
    {
        "question": "What is the difference between semisupervised learning and fully supervised learning?",
        "category": "comparison",
        "expected_pages": [16, 21],
        "reference_answer": "Fully supervised learning requires labels for every training example, while semisupervised learning works with mostly unlabeled data plus just a small amount of labeled data, as in photo-hosting services that only need one label per person."
    },
    {
        "question": "What is the difference between anomaly detection and novelty detection?",
        "category": "comparison",
        "expected_pages": [20, 21],
        "reference_answer": "Anomaly detection algorithms are usually tolerant of a small percentage of outliers in the training data and try to detect unusual instances such as fraud; novelty detection algorithms instead expect to see only normal data during training, so they can flag any genuinely new pattern as novel."
    },
    {
        "question": "How does batch learning differ from Reinforcement Learning in terms of how the system improves?",
        "category": "comparison",
        "expected_pages": [22, 23, 24],
        "reference_answer": "Batch learning improves only by being retrained from scratch on the full, updated dataset; a Reinforcement Learning agent instead improves by trial and error, observing the environment, taking actions, and updating its policy based on the rewards or penalties it receives."
    },
    {
        "question": "What is the difference between a model parameter and a hyperparameter?",
        "category": "comparison",
        "expected_pages": [28, 38],
        "reference_answer": "A model parameter (such as theta0 and theta1 in a linear model) is a value the learning algorithm tunes during training to fit the data; a hyperparameter is a parameter of the learning algorithm itself, set before training and held constant, such as the amount of regularization to apply."
    },
    {
        "question": "Between bad data and a bad algorithm, which does the book suggest is more often the real limiting factor, and why?",
        "category": "comparison",
        "expected_pages": [32, 33],
        "reference_answer": "The book highlights the \"Unreasonable Effectiveness of Data\" research showing that very different algorithms, even fairly simple ones, performed almost identically well on a complex problem once given enough data -- suggesting that more or better data can matter as much as, or more than, a better algorithm."
    },
    {
        "question": "How does increasing regularization affect the trade-off between overfitting and underfitting?",
        "category": "comparison",
        "expected_pages": [37, 38],
        "reference_answer": "Increasing the regularization hyperparameter constrains the model more, reducing overfitting, but if set too high the model becomes too constrained and can underfit instead -- an almost flat model that is unlikely to find a good solution."
    },
    {
        "question": "How does online learning differ from batch learning when dealing with huge datasets that don't fit in memory?",
        "category": "comparison",
        "expected_pages": [24, 25],
        "reference_answer": "Online learning can handle huge datasets through out-of-core learning, loading and training on part of the data at a time and repeating until all of it has been processed, while batch learning needs the entire dataset available at once, which can be impractical at very large scale."
    },

    # ---------------- MULTI-PART / REASONING (10) ----------------
    {
        "question": "Walk through the typical Machine Learning project pipeline as described in the chapter, from getting data to launching the system, and mention one thing that can go wrong at each stage.",
        "category": "multi_part",
        "expected_pages": [13, 32],
        "reference_answer": "A project generally involves gathering training data, training a model, and evaluating/launching it; things that can go wrong include insufficient or poor-quality data, a nonrepresentative training set (sampling bias), irrelevant features, and an algorithm that overfits or underfits the data."
    },
    {
        "question": "Explain how hyperparameters, regularization, and overfitting relate to each other.",
        "category": "multi_part",
        "expected_pages": [37, 38],
        "reference_answer": "Regularization constrains a model to reduce overfitting, and the amount of regularization applied is itself a hyperparameter set before training; tuning that hyperparameter trades off between overfitting (too little regularization) and underfitting (too much, producing an overly flat model)."
    },
    {
        "question": "Compare all four categories of learning based on supervision, and explain which category online learning could belong to.",
        "category": "multi_part",
        "expected_pages": [16, 24],
        "reference_answer": "The four categories are supervised (labeled data), unsupervised (unlabeled data), semisupervised (mostly unlabeled, some labeled), and Reinforcement Learning (rewards/penalties); online learning is a separate criterion describing how a system learns over time (incrementally) rather than what kind of labels it uses, so an online learning system can in principle belong to any of the four supervision categories."
    },
    {
        "question": "The book discusses both 'bad data' and 'bad algorithm' as reasons ML projects fail. Summarize the data-related problems it lists, and explain how this connects to the idea that more data can matter more than a better algorithm.",
        "category": "multi_part",
        "expected_pages": [32, 33],
        "reference_answer": "Data-related problems include insufficient quantity of training data and nonrepresentative training data; the book connects this to the \"Unreasonable Effectiveness of Data\" research, where different algorithms performed comparably well once given enough data, suggesting data quantity/quality can matter as much as algorithm choice."
    },
    {
        "question": "Explain why a model that performs very well on the training set can still fail in production, using the ideas of overfitting, the test set, and generalization error together.",
        "category": "multi_part",
        "expected_pages": [36, 39],
        "reference_answer": "A model can overfit the training set, fitting noise and quirks specific to that data rather than generalizable patterns, so it scores well on training data; the test set measures the generalization error on unseen data, revealing the gap between training performance and real-world performance."
    },
    {
        "question": "Explain how test sets, validation sets, and cross-validation work together to help you pick the right model and hyperparameters without overfitting to the test data.",
        "category": "multi_part",
        "expected_pages": [39, 40, 41],
        "reference_answer": "The validation set (or cross-validation across several folds of the training data) is used to compare candidate models and tune hyperparameters, so decisions are never based on the test set; the test set is then used only once, at the very end, to give an unbiased estimate of generalization performance."
    },
    {
        "question": "Contrast instance-based and model-based learning, and explain which one relates more closely to an algorithm like k-Nearest Neighbors versus fitting a linear regression model.",
        "category": "multi_part",
        "expected_pages": [26, 27, 31],
        "reference_answer": "Instance-based learning generalizes by comparing new cases to stored examples using a similarity measure, which is how k-Nearest Neighbors works; model-based learning builds a general model, such as a Linear Regression equation with tuned parameters, and uses that model to predict on new inputs."
    },
    {
        "question": "Explain sampling bias and nonresponse bias together with an example from the book, and describe how they can affect a Machine Learning system's conclusions.",
        "category": "multi_part",
        "expected_pages": [35],
        "reference_answer": "Sampling bias happens when the training data isn't representative of the real population; nonresponse bias is one cause, illustrated by the 1936 Literary Digest poll, where wealthier, more politically engaged people were overrepresented -- in both cases, the trained model or conclusion ends up systematically skewed for underrepresented groups."
    },
    {
        "question": "Describe how Reinforcement Learning differs from supervised learning in terms of what feedback the system receives, and give the example the book uses.",
        "category": "multi_part",
        "expected_pages": [22, 23],
        "reference_answer": "In supervised learning the system is given explicit correct labels for each example, while in Reinforcement Learning the agent instead receives rewards or penalties for its actions and must learn a policy through trial and error; the book uses the example of DeepMind's AlphaGo, which learned its winning policy by analyzing millions of games and then playing against itself."
    },
    {
        "question": "Summarize the main challenges of Machine Learning the chapter identifies, grouping them into problems with the data versus problems with the algorithm.",
        "category": "multi_part",
        "expected_pages": [32, 36, 38],
        "reference_answer": "Data-related challenges include insufficient quantity of training data, nonrepresentative training data (sampling bias), and poor-quality or irrelevant features; algorithm-related challenges are overfitting the training data and underfitting the training data."
    },

    # ---------------- OUT-OF-DOCUMENT (10) ----------------
    {
        "question": "According to the PDF, what is the capital of Japan?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document rather than answering from general knowledge."
    },
    {
        "question": "What is the boiling point of water at sea level?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "Who wrote the play Romeo and Juliet?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "What year did World War II end?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "What is the capital of Australia?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "Can you recommend a good recipe for making pasta?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF, and it isn't a Machine Learning question either. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "What is today's stock price of Apple Inc.?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF, and the chatbot has no access to live/real-time data. It should say the information was not found in the provided document."
    },
    {
        "question": "Who is the current President of the United States?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "What is the chemical formula for table salt?",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document."
    },
    {
        "question": "Translate the word 'hello' into French.",
        "category": "out_of_document",
        "expected_pages": [],
        "reference_answer": "This is not covered in the PDF. The chatbot should say the information was not found in the provided document rather than performing an unrelated translation task."
    },
]

print(f"Evaluation questions: {len(evaluation_data)}")
for cat in ["factual", "comparison", "multi_part", "out_of_document"]:
    count = sum(1 for item in evaluation_data if item["category"] == cat)
    print(f"  {cat}: {count}")


Evaluation questions: 60
  factual: 25
  comparison: 15
  multi_part: 10
  out_of_document: 10


In [21]:
# 1) Retrieval evaluation

retrieval_rows = []
unverified_count = 0

for item in evaluation_data:
    docs = safe_retrieve(item["question"])

    retrieved_pages = sorted({
        doc.metadata.get("page")
        for doc in docs
        if doc.metadata.get("page") is not None
    })

    raw_expected = item.get("expected_pages")

    if raw_expected is None:
        # Page numbers not verified against the PDF yet -- skip this
        # row for the hit-rate calculation instead of guessing.
        expected_pages = []
        hit = None
        unverified_count += 1
    elif len(raw_expected) == 0:
        # Out-of-document question: there is no expected source page.
        expected_pages = []
        hit = None
    else:
        expected_pages = sorted(set(raw_expected))
        hit = bool(set(retrieved_pages).intersection(expected_pages))

    retrieval_rows.append({
        "question": item["question"],
        "category": item.get("category", "uncategorized"),
        "expected_pages": expected_pages,
        "retrieved_pages": retrieved_pages,
        "hit": hit
    })

retrieval_df = pd.DataFrame(retrieval_rows)

scored = retrieval_df[retrieval_df["hit"].notna()]
retrieval_hit_rate = scored["hit"].mean() if len(scored) else 0.0

print(f"Retrieval Hit Rate: {retrieval_hit_rate:.2%}  ({len(scored)} questions with a verified source page)")
if unverified_count:
    print(f"{unverified_count} question(s) have expected_pages=None (not verified yet) and are excluded from this rate.")

print("\nRetrieval Hit Rate by category (out_of_document has no source page, so it's excluded):")
display(scored.groupby("category")["hit"].mean().apply(lambda x: f"{x:.2%}"))

display(retrieval_df[["category", "question", "expected_pages", "retrieved_pages", "hit"]])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieval Hit Rate: 76.00%  (50 questions with a verified source page)

Retrieval Hit Rate by category (out_of_document has no source page, so it's excluded):


category
comparison    86.67%
factual       64.00%
multi_part    90.00%
Name: hit, dtype: object

,category,question,expected_pages,retrieved_pages,hit
0,factual,What is Machine Learning?,[12],"[11, 12, 15, 16, 39, 41, 46, 159]",True
1,factual,What is a training set?,[12],"[16, 39, 42, 63, 123, 149, 216, 227]",False
2,factual,Why is Machine Learning useful for problems th...,"[13, 14]","[14, 15, 33, 38, 41, 123, 184]",True
3,factual,What are the main criteria used to classify Ma...,[15],"[11, 17, 26, 27, 41, 171, 184]",False
4,factual,What are the four major categories based on th...,[16],"[16, 41, 43, 75, 190, 220, 258, 272]",True
5,factual,What is supervised learning?,[16],"[11, 16, 18, 22, 113, 238, 243]",True
6,factual,What is a typical classification task mentione...,[16],"[11, 21, 41, 113, 117, 171, 190, 258]",False
7,factual,What is regression in supervised learning?,"[16, 17]","[16, 47, 72, 113, 118, 119, 189]",True
8,factual,What is unsupervised learning?,[18],"[16, 18, 22, 25, 224, 241, 242, 279]",True
9,factual,Name some unsupervised learning algorithms dis...,[18],"[15, 16, 17, 33, 124, 241, 242, 279]",False


In [22]:
# 2) Inspect retrieved chunks for any question

def inspect_retrieval(question, n=8):
    docs = retriever.invoke(question)

    print("=" * 90)
    print("QUESTION:", question)
    print("=" * 90)

    for i, doc in enumerate(docs[:n], 1):
        print(f"\n--- Retrieved chunk {i} ---")
        print("Page:", doc.metadata.get("page"))
        print("Chapter:", doc.metadata.get("chapter"))
        print(doc.page_content[:700])

# Example:
# inspect_retrieval("What is supervised learning?")


### 3) Answer evaluation

This uses the same local Ollama model as the judge. It is not independent of the answering model, so treat scores as indicative. It checks:

- **Correctness** — does the answer address the question?
- **Groundedness** — is the answer supported by the retrieved PDF context?
- **Overall score**
  - `2` = correct and supported
  - `1` = partially correct / partially supported
  - `0` = incorrect or unsupported

This is an evaluation signal, not a mathematically exact accuracy measurement.

### Metric denominator audit and Row 0

The saved notebook outputs below are historical: retrieval was 38/50 answerable questions (76.00%); answer score averaged 85/57 valid judgments (1.49/2.00), with 3 of 60 unscored; 32/57 valid judgments were fully correct and grounded (56.14%). The old saved printout shows 53.33% because it divided those 32 successes by all 60 rows. The corrected summary code now uses the valid-score denominator. The judge was the same `llama3.2:3b` model used to generate answers.

The measured 4096-token Row 0 is 38/50 retrieval hits (76.00%); 85/51 average answer score (1.67/2.00); and 38/51 fully correct and grounded (74.51%). Nine judgment outputs were invalid and excluded. Average retrieval was 0.804 s, time to first token 43.015 s, and generation 65.405 s, each averaged over 60 questions. Full output: `evaluation/results/row0.json`.


In [23]:
judge_prompt = ChatPromptTemplate.from_template(r"""
You are evaluating a RAG chatbot answer.

Use ONLY the provided PDF context and reference answer.

Question:
{question}

Reference answer:
{reference_answer}

Retrieved PDF context:
{context}

Chatbot answer:
{answer}

Score the chatbot answer yourself. Reply with ONLY a JSON object with these
four keys: correctness (0, 1, or 2), groundedness (0, 1, or 2), overall_score
(0, 1, or 2), and reason (a short sentence explaining YOUR OWN judgment of
THIS SPECIFIC answer -- do not use generic placeholder text).

Scoring:
0 = incorrect or unsupported
1 = partially correct or partially supported
2 = correct and supported

For an out-of-document question, a good answer should explicitly say that the information was not found in the provided PDF/context rather than inventing an answer.
""")

# Reasons a lazy/weak judge model sometimes falls back to instead of
# actually scoring -- if we see one of these verbatim, treat the score
# as missing (None) rather than a real 0, so it does not silently drag
# down the average.
_PLACEHOLDER_REASONS = {"short explanation", "explanation", ""}

def parse_judge_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        return {
            "correctness": None,
            "groundedness": None,
            "overall_score": None,
            "reason": text
        }

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        # Small models often emit almost-valid JSON (e.g. an unescaped quote in
        # "reason"). Pull the numeric scores out directly instead of losing them.
        found = {}
        for key in ("correctness", "groundedness", "overall_score"):
            m = re.search(r'"%s"\s*:\s*([0-2])' % key, text)
            if m:
                found[key] = int(m.group(1))
        if "correctness" in found and "groundedness" in found:
            found.setdefault("overall_score", min(found["correctness"], found["groundedness"]))
            found["reason"] = "scores recovered from malformed judge JSON"
            return found
        return {
            "correctness": None,
            "groundedness": None,
            "overall_score": None,
            "reason": text
        }

    reason = str(parsed.get("reason", "")).strip().lower()
    if reason in _PLACEHOLDER_REASONS:
        # The judge just echoed the template instead of scoring --
        # mark this as unscored, not a genuine 0.
        return {
            "correctness": None,
            "groundedness": None,
            "overall_score": None,
            "reason": "judge did not produce a real score (template echo)"
        }

    return parsed


In [24]:
# Run answer evaluation.
# This runs the full chatbot pipeline + judge locally via Ollama for every
# question below (2 model calls each) -- with 60 questions on an 8GB CPU-only
# setup this can take a long time (multiple hours).
#
# CRASH-RESUMABLE: results are saved to CHECKPOINT_FILE after every single
# question. If this cell (or the kernel, or Ollama) crashes partway through,
# just run this cell again -- it will pick up where it left off instead of
# starting over from question 1. To force a full restart, delete the
# CHECKPOINT_FILE file.
#
# Each question also gets its own timeout so a single hung Ollama call can't
# freeze the whole run silently for hours -- it will time out, get logged as
# an error row, and the loop moves on to the next question.

import os
import time
import csv
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError

# The checkpoint name includes a fingerprint of every setting/prompt/question that
# affects the results, so changing any of them automatically starts a fresh run
# instead of silently re-using stale scores from an older configuration.
CONFIG_SIG = hashlib.sha256("|".join([
    EMBEDDING_MODEL, OLLAMA_MODEL, str(TOP_K), str(FETCH_K), str(MMR_LAMBDA),
    str(CHUNK_SIZE), str(CHUNK_OVERLAP), repr(answer_prompt), repr(judge_prompt),
    json.dumps([q["question"] for q in evaluation_data]),
]).encode()).hexdigest()[:8]
CHECKPOINT_FILE = f"answer_eval_checkpoint_{PDF_HASH}_{CONFIG_SIG}.csv"
PER_QUESTION_TIMEOUT = 420  # seconds -- long answers on a 3B CPU model can take several minutes

CHECKPOINT_FIELDS = [
    "idx", "question", "category", "answer",
    "correctness", "groundedness", "overall_score", "reason"
]

# Fresh checkpointer: evaluation questions never share conversational memory.
evaluation_app = workflow.compile(checkpointer=MemorySaver())

# Load already-completed rows from a previous (possibly crashed) run.
done_indices = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", newline="", encoding="utf-8") as f:
        saved_rows = list(csv.DictReader(f))
    # Rows that timed out or errored are retried, not treated as finished.
    good_rows = [r for r in saved_rows
                 if not str(r.get("reason", "")).startswith(("TIMEOUT", "ERROR"))]
    with open(CHECKPOINT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CHECKPOINT_FIELDS)
        writer.writeheader()
        writer.writerows(good_rows)
    done_indices = {int(r["idx"]) for r in good_rows}
    print(f"Resuming: {len(done_indices)} question(s) already completed in {CHECKPOINT_FILE}; "
          f"{len(saved_rows) - len(good_rows)} failed row(s) will be retried.")
else:
    with open(CHECKPOINT_FILE, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CHECKPOINT_FIELDS).writeheader()


def run_one_question(item, idx):
    result = evaluation_app.invoke(
        {"messages": [HumanMessage(content=item["question"])]},
        config={"configurable": {"thread_id": f"evaluation-{PDF_HASH}-{idx}"}}
    )
    answer = result.get("answer", "")
    docs = result.get("retrieved_docs", [])
    context = format_context(docs)

    judge_messages = judge_prompt.format_messages(
        question=item["question"],
        reference_answer=item["reference_answer"],
        context=context,
        answer=answer
    )
    judge_response = llm.invoke(judge_messages)
    judge_text = clean_response(judge_response)
    scores = parse_judge_json(judge_text)

    return {
        "idx": idx,
        "question": item["question"],
        "category": item.get("category", "uncategorized"),
        "answer": answer,
        "correctness": scores.get("correctness"),
        "groundedness": scores.get("groundedness"),
        "overall_score": scores.get("overall_score"),
        "reason": scores.get("reason")
    }


total = len(evaluation_data)
executor = ThreadPoolExecutor(max_workers=1)

for idx, item in enumerate(evaluation_data, 1):
    if idx in done_indices:
        continue

    start = time.time()
    print(f"[{idx}/{total}] {item['question'][:60]!r} ...", end=" ", flush=True)

    try:
        future = executor.submit(run_one_question, item, idx)
        row = future.result(timeout=PER_QUESTION_TIMEOUT)
        print(f"done in {time.time() - start:.0f}s (score={row['overall_score']})")
    except FutureTimeoutError:
        row = {
            "idx": idx, "question": item["question"],
            "category": item.get("category", "uncategorized"), "answer": "",
            "correctness": None, "groundedness": None, "overall_score": None,
            "reason": f"TIMEOUT after {PER_QUESTION_TIMEOUT}s"
        }
        print(f"TIMEOUT after {PER_QUESTION_TIMEOUT}s -- logged and continuing")
    except Exception as e:
        row = {
            "idx": idx, "question": item["question"],
            "category": item.get("category", "uncategorized"), "answer": "",
            "correctness": None, "groundedness": None, "overall_score": None,
            "reason": f"ERROR: {e}"
        }
        print(f"ERROR: {e} -- logged and continuing")

    # Save immediately -- this question's result survives even if the
    # next one crashes the kernel.
    with open(CHECKPOINT_FILE, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CHECKPOINT_FIELDS).writerow(row)

executor.shutdown(wait=False)

answer_eval_df = pd.read_csv(CHECKPOINT_FILE)

print("\nAnswer evaluation completed.")
display(answer_eval_df[
    ["category", "question", "correctness", "groundedness", "overall_score", "reason"]
])


[1/60] 'What is Machine Learning?' ... done in 115s (score=2)
[2/60] 'What is a training set?' ... done in 75s (score=2)
[3/60] 'Why is Machine Learning useful for problems that require man' ... done in 96s (score=1)
[4/60] 'What are the main criteria used to classify Machine Learning' ... done in 74s (score=1)
[5/60] 'What are the four major categories based on the amount and t' ... done in 70s (score=2)
[6/60] 'What is supervised learning?' ... done in 123s (score=1)
[35/60] 'What is the difference between anomaly detection and novelty' ... done in 134s (score=1)
[36/60] 'How does batch learning differ from Reinforcement Learning i' ... done in 126s (score=1)
[37/60] 'What is the difference between a model parameter and a hyper' ... done in 117s (score=1)
[38/60] 'Between bad data and a bad algorithm, which does the book su' ... done in 93s (score=2)
[39/60] 'How does increasing regularization affect the trade-off betw' ... done in 119s (score=1)
[40/60] 'How does online learning dif

,category,question,correctness,groundedness,overall_score,reason
0,factual,What is Machine Learning?,1.0,2.0,2.0,"The answer is partially correct, as it referen..."
1,factual,What is a training set?,2.0,2.0,2.0,The chatbot answer accurately defines a traini...
2,factual,Why is Machine Learning useful for problems th...,1.0,2.0,1.0,"The answer is partially correct, as it mention..."
3,factual,What are the main criteria used to classify Ma...,0.0,1.0,1.0,"The answer partially corrects the question, bu..."
4,factual,What are the four major categories based on th...,1.0,2.0,2.0,The answer is partially correct because it lis...
5,factual,What is supervised learning?,2.0,2.0,2.0,The answer accurately defines supervised learn...
6,factual,What is a typical classification task mentione...,0.0,0.0,0.0,The answer is not supported by the provided PD...
7,factual,What is regression in supervised learning?,2.0,2.0,2.0,The answer accurately quotes the PDF context a...
8,factual,What is unsupervised learning?,2.0,2.0,2.0,The answer is correct and supported by the pro...
9,factual,Name some unsupervised learning algorithms dis...,0.0,1.0,1.0,"The answer mentions k-Nearest Neighbors, which..."


In [25]:
# 4) Evaluation summary

answer_eval_df["overall_score_numeric"] = pd.to_numeric(
    answer_eval_df["overall_score"], errors="coerce"
)
numeric_scores = answer_eval_df["overall_score_numeric"]

unscored = numeric_scores.isna().sum()

print(f"Questions evaluated: {len(answer_eval_df)}")
print(f"Retrieval Hit Rate: {retrieval_hit_rate:.2%} ({len(scored)} answerable questions)")

if unscored:
    print(f"Judge could not produce a real score for {unscored} question(s) "
          f"(excluded from the averages below, not counted as 0).")

valid_scores = numeric_scores.dropna()
if len(valid_scores):
    print(f"Average Answer Score: {valid_scores.mean():.2f} / 2.00 ({len(valid_scores)} valid judge outputs)")
    print(
        f"Fully correct & grounded: "
        f"{(valid_scores == 2).mean():.2%} ({(valid_scores == 2).sum()}/{len(valid_scores)} valid judge outputs)"
    )

print("\nAverage Answer Score by category:")
display(
    answer_eval_df.groupby("category")["overall_score_numeric"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_score", "count": "n"})
)

print("\nRetrieval Hit Rate and answer scores measure different parts of the RAG pipeline.")


Questions evaluated: 60
Retrieval Hit Rate: 76.00%
Judge could not produce a real score for 3 question(s) (excluded from the averages below, not counted as 0).
Average Answer Score: 1.49 / 2.00
Fully correct & grounded: 53.33%

Average Answer Score by category:


,avg_score,n
category,,
comparison,1.428571,14
factual,1.458333,24
multi_part,1.666667,9
out_of_document,1.500000,10



Retrieval Hit Rate and answer scores measure different parts of the RAG pipeline.


## How to use another PDF

You do **not** need to change the RAG pipeline.

1. Put the new PDF in the notebook folder.
2. Change only:
   ```python
   PDF_PATH = "your_new_file.pdf"
   ```
3. Run all cells from the top.
4. The PDF content hash creates a separate ChromaDB directory.
5. The ingestion pipeline cleans every chunk before it reaches the tokenizer.
6. Run the chatbot normally.
7. For evaluation, replace `evaluation_data` with questions and expected pages from the new PDF.

### Important

The evaluation questions included in this notebook are based on the uploaded ML ebook. They are **not** hard-coded into the chatbot itself.

For a different PDF, the chatbot remains generic; only the evaluation dataset should be changed to match that document.
